# 11_graph_model_input_packaging.ipynb

## Στόχος

Το παρόν notebook λειτουργεί ως **strict graph-model input packaging stage** πάνω στα ήδη επαληθευμένα canonical artifacts του forecasting pipeline.

Πιο συγκεκριμένα, το notebook:

- φορτώνει τα canonical split artifacts:
  - `train_final.csv`
  - `val_final.csv`
  - `test_final.csv`
- φορτώνει τα canonical graph artifacts:
  - `graph_node_order.csv`
  - `graph_edge_index.npy`
  - `graph_distance_matrix_km.npy`
- επιβεβαιώνει ξανά το split-to-graph packaging contract,
- ορίζει ρητά:
  - backbone / control columns,
  - target column,
  - static node features,
  - dynamic node features,
  - excluded / blocked columns,
- και εξάγει graph-model-ready packaged datasets χωρίς training.

## Τι κάνει το NB11

Το NB11 περιορίζεται σε:

1. deterministic loading των canonical artifacts,
2. strict canonical normalization του `park_id` ως string,
3. strict canonical parsing του `timestamp` μόνο στο επίπεδο των downstream split artifacts,
4. explicit feature-role definition,
5. graph-ready tensor packaging ανά split,
6. intentional export manifests και serialized graph-ready packages.

## Τι δεν κάνει το NB11

Το NB11:

- **δεν** κάνει raw validation,
- **δεν** ξανανοίγει responsibilities του `NB02`,
- **δεν** αλλάζει το canonical split contract,
- **δεν** αλλάζει benchmark reporting logic,
- **δεν** αγγίζει το `data/processed/baseline_metrics.csv`,
- **δεν** κάνει GNN training,
- **δεν** κάνει graph benchmarking,
- **δεν** εισάγει PHM / anomaly / fault claims.

## Μεθοδολογική θέση

Το notebook είναι **forecasting-first, benchmark-safe, packaging-only bridge stage** μετά το `NB10`.

Δεν αποτελεί ακόμη graph-training notebook και δεν προσφέρει graph-model evidence.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import importlib
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

# Προαιρετικό import για PyTorch Geometric preview objects.
try:
    from torch_geometric.data import Data
    TORCH_PYG_AVAILABLE = True
except Exception:
    Data = None
    TORCH_PYG_AVAILABLE = False


# ============================================================
# Project-root / path helpers
# ============================================================

def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το repository root ανεβαίνοντας από το current working directory.
    """
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε project root με φακέλους `data/` και `notebooks/`."
    )


def resolve_under_root(path_like: str | Path, root: Path) -> Path:
    """
    Αν το path είναι σχετικό, το επιλύουμε κάτω από το project root.
    Αν είναι ήδη absolute, το επιστρέφουμε ως έχει.
    """
    path_obj = Path(path_like)
    return path_obj if path_obj.is_absolute() else (root / path_obj)


ROOT = find_project_root(Path.cwd())

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))


# ============================================================
# Προαιρετική φόρτωση src.config
# ============================================================

try:
    cfg = importlib.import_module("src.config")
except Exception:
    cfg = None


def cfg_get(name: str, default: Any) -> Any:
    """
    Διαβάζει attribute από src.config μόνο αν υπάρχει.
    """
    if cfg is None:
        return default
    return getattr(cfg, name, default)


# ============================================================
# Canonical constants από config
# ============================================================

PARK_ID_COLUMN = cfg_get("PARK_ID_COLUMN", "park_id")
TIMESTAMP_COLUMN = cfg_get("TIMESTAMP_COLUMN", "timestamp")
TARGET_COLUMN = cfg_get("TARGET_COLUMN", "Power_Output_Normalized")
BASELINE_COLUMN = cfg_get("BASELINE_COLUMN", "Baseline_Prediction")
TEST_FLAG_COLUMN = cfg_get("TEST_FLAG_COLUMN", "test_flag")
NODE_IDX_COLUMN = "node_idx"

DATA_PROCESSED_DIR = resolve_under_root(
    cfg_get("DATA_PROCESSED", "data/processed"),
    ROOT,
)

TRAIN_PATH = resolve_under_root(
    cfg_get("TRAIN_FINAL_PATH", "data/processed/train_final.csv"),
    ROOT,
)
VAL_PATH = resolve_under_root(
    cfg_get("VAL_FINAL_PATH", "data/processed/val_final.csv"),
    ROOT,
)
TEST_PATH = resolve_under_root(
    cfg_get("TEST_FINAL_PATH", "data/processed/test_final.csv"),
    ROOT,
)

GRAPH_NODE_ORDER_PATH = resolve_under_root(
    cfg_get("GRAPH_NODE_ORDER_PATH", "data/processed/graph_node_order.csv"),
    ROOT,
)
GRAPH_EDGE_INDEX_PATH = resolve_under_root(
    cfg_get("GRAPH_EDGE_INDEX_PATH", "data/processed/graph_edge_index.npy"),
    ROOT,
)
GRAPH_DISTANCE_MATRIX_PATH = resolve_under_root(
    cfg_get("GRAPH_DISTANCE_MATRIX_PATH", "data/processed/graph_distance_matrix_km.npy"),
    ROOT,
)

EXPORT_DIR = resolve_under_root(
    cfg_get("NB11_EXPORT_DIR", "data/processed/graph_packaging"),
    ROOT,
)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

NB11_FEATURE_ROLE_MANIFEST_PATH = resolve_under_root(
    cfg_get("NB11_FEATURE_ROLE_MANIFEST", EXPORT_DIR / "nb11_feature_role_manifest.csv"),
    ROOT,
)
NB11_NODE_FEATURE_MANIFEST_PATH = resolve_under_root(
    cfg_get("NB11_NODE_FEATURE_MANIFEST", EXPORT_DIR / "nb11_node_feature_manifest.csv"),
    ROOT,
)
NB11_SPLIT_GRAPH_PACKAGING_SUMMARY_PATH = resolve_under_root(
    cfg_get(
        "NB11_SPLIT_GRAPH_PACKAGING_SUMMARY",
        EXPORT_DIR / "nb11_split_graph_packaging_summary.csv",
    ),
    ROOT,
)
NB11_PACKAGING_STATUS_MANIFEST_PATH = resolve_under_root(
    cfg_get(
        "NB11_PACKAGING_STATUS_MANIFEST",
        EXPORT_DIR / "nb11_packaging_status_manifest.csv",
    ),
    ROOT,
)

NB11_TRAIN_GRAPH_DATASET_PATH = resolve_under_root(
    cfg_get("NB11_TRAIN_GRAPH_DATASET", EXPORT_DIR / "train_graph_dataset.pt"),
    ROOT,
)
NB11_VAL_GRAPH_DATASET_PATH = resolve_under_root(
    cfg_get("NB11_VAL_GRAPH_DATASET", EXPORT_DIR / "val_graph_dataset.pt"),
    ROOT,
)
NB11_TEST_GRAPH_DATASET_PATH = resolve_under_root(
    cfg_get("NB11_TEST_GRAPH_DATASET", EXPORT_DIR / "test_graph_dataset.pt"),
    ROOT,
)
NB11_PREVIEW_PYG_OBJECTS_PATH = resolve_under_root(
    cfg_get("NB11_PREVIEW_PYG_OBJECTS", EXPORT_DIR / "nb11_preview_pyg_objects.pt"),
    ROOT,
)

WRITE_SERIALIZED_PACKAGES = True
EXPORT_PYG_PREVIEW_OBJECTS = True

# Υποψήφιες static numeric στήλες.
# Αν κάποια δεν υπάρχει στο actual schema, δεν τη χρησιμοποιούμε σιωπηλά.
STATIC_NUMERIC_CANDIDATES = [
    "lat",
    "long",
    "hub_height_m",
    "rotor_diameter_m",
    "nominal_power_kW",
]

# Υποψήφιες static non-numeric στήλες που μπλοκάρονται από default packaging.
STATIC_NON_NUMERIC_CANDIDATES = [
    "turbine",
]

REQUIRED_SPLIT_COLUMNS = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}

print("Project root:", ROOT)
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:", VAL_PATH)
print("TEST_PATH:", TEST_PATH)
print("GRAPH_NODE_ORDER_PATH:", GRAPH_NODE_ORDER_PATH)
print("GRAPH_EDGE_INDEX_PATH:", GRAPH_EDGE_INDEX_PATH)
print("GRAPH_DISTANCE_MATRIX_PATH:", GRAPH_DISTANCE_MATRIX_PATH)
print("EXPORT_DIR:", EXPORT_DIR)
print("TORCH_PYG_AVAILABLE:", TORCH_PYG_AVAILABLE)

Project root: C:\Users\diony\Desktop\WindPower_DigitalTwin
TRAIN_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\train_final.csv
VAL_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\val_final.csv
TEST_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\test_final.csv
GRAPH_NODE_ORDER_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_node_order.csv
GRAPH_EDGE_INDEX_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_edge_index.npy
GRAPH_DISTANCE_MATRIX_PATH: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_distance_matrix_km.npy
EXPORT_DIR: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging
TORCH_PYG_AVAILABLE: True


## Explicit assumptions

Το NB11 δεν ξανακάνει raw validation.

Υποθέτουμε ότι τα canonical split artifacts και graph artifacts έχουν ήδη παραχθεί και verified upstream.

### Assumption note για ονόματα στηλών

Για ορισμένα static metadata / spatial columns χρησιμοποιούνται **candidate names** όπως:

- `lat`
- `long`
- `hub_height_m`
- `rotor_diameter_m`
- `nominal_power_kW`
- `turbine`

Αν κάποιο από αυτά δεν υπάρχει στο actual schema, το notebook **δεν** το εφευρίσκει και **δεν** το υποκαθιστά σιωπηλά.
Αντίθετα, το καταγράφει ρητά μέσω defensive checks και manifests.

### Design choice του NB11

Το notebook παράγει δύο packaging layers:

1. **Serialized split packages** για future model code
2. **PyTorch Geometric–ready preview snapshots** για deterministic inspection

### Boundary note

Το notebook δεν λύνει ακόμη:

- static categorical encoding policy για `turbine`,
- temporal batching strategy για training,
- graph-sequence architecture design,
- memory-optimized dataloader implementation,
- graph benchmark evaluation.

Αυτά ανήκουν σε επόμενο modeling stage, όχι στο current packaging stage.

In [2]:
# ============================================================
# Canonical normalization / validation helpers
# ============================================================

def standardize_park_id(series: pd.Series) -> pd.Series:
    """
    Μετατρέπει το park_id σε canonical zero-padded string identifier.
    """
    out = series.astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.str.zfill(5)

    if out.isna().any():
        raise ValueError("Βρέθηκαν null park_id values μετά το normalization.")

    return out


def parse_canonical_split_timestamp(series: pd.Series) -> pd.Series:
    """
    Strict parsing μόνο για τα canonical split artifacts.
    Δεν αποτελεί raw validation logic.
    """
    parsed = pd.to_datetime(
        series,
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )

    if parsed.isna().any():
        raise ValueError("Βρέθηκαν NaT timestamps στα canonical split artifacts.")

    return parsed


def ensure_required_columns(df: pd.DataFrame, required_cols: set[str], df_name: str) -> None:
    """
    Ελέγχει αν το dataframe περιέχει όλες τις required στήλες.
    """
    missing = sorted(required_cols - set(df.columns))
    if missing:
        raise KeyError(f"Λείπουν required columns από `{df_name}`: {missing}")


def assert_no_duplicate_keys(df: pd.DataFrame, key_cols: list[str], df_name: str) -> None:
    """
    Ελέγχει duplicate rows πάνω στο canonical key-space.
    """
    dup_count = int(df.duplicated(subset=key_cols).sum())
    if dup_count > 0:
        raise ValueError(
            f"Βρέθηκαν {dup_count} duplicate rows στο `{df_name}` "
            f"πάνω στα keys {key_cols}."
        )


def assert_no_overlap(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    key_cols: list[str],
    left_name: str,
    right_name: str,
) -> None:
    """
    Ελέγχει ότι δύο splits δεν έχουν overlap στο canonical key-space.
    """
    overlap = left_df[key_cols].merge(right_df[key_cols], on=key_cols, how="inner")
    if len(overlap) > 0:
        raise ValueError(
            f"Βρέθηκε overlap μεταξύ `{left_name}` και `{right_name}` "
            f"στο backbone key space {key_cols}. overlap_rows={len(overlap)}"
        )


def load_split(csv_path: Path, split_name: str) -> pd.DataFrame:
    """
    Φορτώνει canonical split artifact με strict downstream checks.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Λείπει split artifact: {csv_path}")

    df = pd.read_csv(csv_path, low_memory=False)

    ensure_required_columns(df, REQUIRED_SPLIT_COLUMNS, split_name)

    if df.columns.duplicated().any():
        dup_cols = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate columns στο `{split_name}`: {dup_cols}")

    df = df.copy()
    df[PARK_ID_COLUMN] = standardize_park_id(df[PARK_ID_COLUMN])
    df[TIMESTAMP_COLUMN] = parse_canonical_split_timestamp(df[TIMESTAMP_COLUMN])

    if df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN, TEST_FLAG_COLUMN]].isnull().any().any():
        raise ValueError(f"Βρέθηκαν nulls σε core columns του `{split_name}`.")

    df = (
        df.sort_values([PARK_ID_COLUMN, TIMESTAMP_COLUMN], kind="mergesort")
        .reset_index(drop=True)
        .copy()
    )

    assert_no_duplicate_keys(df, [PARK_ID_COLUMN, TIMESTAMP_COLUMN], split_name)

    # Έλεγχος monotonic ordering ανά park.
    bad_parks = []
    for park_id, park_slice in df.groupby(PARK_ID_COLUMN, sort=False):
        if not park_slice[TIMESTAMP_COLUMN].is_monotonic_increasing:
            bad_parks.append(park_id)

    if bad_parks:
        raise ValueError(
            f"Βρέθηκαν parks με non-monotonic timestamp ordering στο `{split_name}`: "
            f"{bad_parks[:10]}"
        )

    return df


def detect_graph_node_id_column(df: pd.DataFrame) -> str:
    """
    Εντοπίζει ποια στήλη παίζει ρόλο park identifier στο graph_node_order.csv.
    """
    candidate_cols = [PARK_ID_COLUMN, "park_id", "Park_ID", "node_id"]

    for col in candidate_cols:
        if col in df.columns:
            return col

    raise KeyError(
        "Δεν βρέθηκε identifier column στο graph_node_order.csv. "
        f"Expected one of: {candidate_cols}"
    )


# ============================================================
# Load canonical split artifacts
# ============================================================

train_df = load_split(TRAIN_PATH, "train")
val_df = load_split(VAL_PATH, "val")
test_df = load_split(TEST_PATH, "test")

if list(train_df.columns) != list(val_df.columns) or list(train_df.columns) != list(test_df.columns):
    raise ValueError("Τα train / val / test splits δεν έχουν identical schema.")

assert_no_overlap(train_df, val_df, [PARK_ID_COLUMN, TIMESTAMP_COLUMN], "train", "val")
assert_no_overlap(train_df, test_df, [PARK_ID_COLUMN, TIMESTAMP_COLUMN], "train", "test")
assert_no_overlap(val_df, test_df, [PARK_ID_COLUMN, TIMESTAMP_COLUMN], "val", "test")

train_flags = set(train_df[TEST_FLAG_COLUMN].dropna().astype(int).unique().tolist())
val_flags = set(val_df[TEST_FLAG_COLUMN].dropna().astype(int).unique().tolist())
test_flags = set(test_df[TEST_FLAG_COLUMN].dropna().astype(int).unique().tolist())

if train_flags != {0}:
    raise ValueError(f"Expected train {TEST_FLAG_COLUMN}={{0}}, found={train_flags}")
if val_flags != {0}:
    raise ValueError(f"Expected val {TEST_FLAG_COLUMN}={{0}}, found={val_flags}")
if test_flags != {1}:
    raise ValueError(f"Expected test {TEST_FLAG_COLUMN}={{1}}, found={test_flags}")


# ============================================================
# Load canonical graph artifacts
# ============================================================

if not GRAPH_NODE_ORDER_PATH.exists():
    raise FileNotFoundError(f"Λείπει graph node-order artifact: {GRAPH_NODE_ORDER_PATH}")
if not GRAPH_EDGE_INDEX_PATH.exists():
    raise FileNotFoundError(f"Λείπει graph edge-index artifact: {GRAPH_EDGE_INDEX_PATH}")
if not GRAPH_DISTANCE_MATRIX_PATH.exists():
    raise FileNotFoundError(f"Λείπει graph distance artifact: {GRAPH_DISTANCE_MATRIX_PATH}")

graph_node_order_df = pd.read_csv(GRAPH_NODE_ORDER_PATH, low_memory=False).copy()
graph_node_id_col = detect_graph_node_id_column(graph_node_order_df)

graph_node_order_df[PARK_ID_COLUMN] = standardize_park_id(graph_node_order_df[graph_node_id_col])

if graph_node_id_col != PARK_ID_COLUMN:
    graph_node_order_df = graph_node_order_df.drop(columns=[graph_node_id_col], errors="ignore")

if NODE_IDX_COLUMN in graph_node_order_df.columns:
    # Αν υπάρχει ήδη node_idx, το ελέγχουμε αντί να το ξαναορίσουμε.
    node_idx_values = pd.to_numeric(graph_node_order_df[NODE_IDX_COLUMN], errors="raise").astype(int)
    if sorted(node_idx_values.tolist()) != list(range(len(graph_node_order_df))):
        raise ValueError(
            "Το existing node_idx στο graph_node_order.csv δεν είναι contiguous 0..N-1."
        )
    graph_node_order_df[NODE_IDX_COLUMN] = node_idx_values
    graph_node_order_df = graph_node_order_df.sort_values(NODE_IDX_COLUMN, kind="mergesort").reset_index(drop=True)
else:
    graph_node_order_df[NODE_IDX_COLUMN] = np.arange(len(graph_node_order_df), dtype=int)

if graph_node_order_df[PARK_ID_COLUMN].duplicated().any():
    dup_ids = graph_node_order_df.loc[
        graph_node_order_df[PARK_ID_COLUMN].duplicated(), PARK_ID_COLUMN
    ].tolist()
    raise ValueError(f"Duplicate park_id values στο graph_node_order.csv: {dup_ids[:10]}")

graph_edge_index = np.asarray(np.load(GRAPH_EDGE_INDEX_PATH))
graph_distance_matrix_km = np.asarray(np.load(GRAPH_DISTANCE_MATRIX_PATH))

if graph_edge_index.ndim != 2 or graph_edge_index.shape[0] != 2:
    raise ValueError(
        "Το graph_edge_index.npy πρέπει να έχει shape [2, E]. "
        f"Found shape={graph_edge_index.shape}"
    )

n_nodes = len(graph_node_order_df)

if graph_distance_matrix_km.shape != (n_nodes, n_nodes):
    raise ValueError(
        "Το graph_distance_matrix_km.npy δεν συμφωνεί με το graph node count. "
        f"distance_shape={graph_distance_matrix_km.shape}, n_nodes={n_nodes}"
    )

if graph_edge_index.min() < 0 or graph_edge_index.max() >= n_nodes:
    raise ValueError(
        f"Out-of-range indices στο edge_index: min={graph_edge_index.min()}, "
        f"max={graph_edge_index.max()}, n_nodes={n_nodes}"
    )

if not np.allclose(graph_distance_matrix_km, graph_distance_matrix_km.T, atol=1e-9):
    raise ValueError("Το distance matrix δεν είναι συμμετρικό.")

if not np.allclose(np.diag(graph_distance_matrix_km), 0.0, atol=1e-9):
    raise ValueError("Η diagonal του distance matrix δεν είναι zero.")

# Επιπλέον graph checks συμβατά με το verification spirit του NB10.
directed_edges = graph_edge_index.T
self_loops = int(np.sum(graph_edge_index[0] == graph_edge_index[1]))
duplicate_directed_edges = int(
    pd.DataFrame(directed_edges, columns=["src", "dst"]).duplicated().sum()
)

if self_loops != 0:
    raise ValueError(f"Βρέθηκαν self-loops στο edge_index: {self_loops}")

if duplicate_directed_edges != 0:
    raise ValueError(f"Βρέθηκαν duplicate directed edges στο edge_index: {duplicate_directed_edges}")


# ============================================================
# Split-to-graph mapping
# ============================================================

node_index_lookup = dict(
    zip(graph_node_order_df[PARK_ID_COLUMN].tolist(), graph_node_order_df[NODE_IDX_COLUMN].tolist())
)
graph_node_set = set(graph_node_order_df[PARK_ID_COLUMN].astype(str).tolist())

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_park_set = set(split_df[PARK_ID_COLUMN].astype(str).tolist())

    missing_from_graph = sorted(split_park_set - graph_node_set)
    if missing_from_graph:
        raise ValueError(
            f"Βρέθηκαν parks στο `{split_name}` που δεν υπάρχουν στο graph_node_order: "
            f"{missing_from_graph[:10]}"
        )

    split_df[NODE_IDX_COLUMN] = split_df[PARK_ID_COLUMN].map(node_index_lookup)

    if split_df[NODE_IDX_COLUMN].isna().any():
        raise ValueError(f"Βρέθηκαν unmapped rows στο `{split_name}` μετά το node mapping.")

    split_df[NODE_IDX_COLUMN] = split_df[NODE_IDX_COLUMN].astype(int)

split_union_park_set = (
    set(train_df[PARK_ID_COLUMN].astype(str))
    | set(val_df[PARK_ID_COLUMN].astype(str))
    | set(test_df[PARK_ID_COLUMN].astype(str))
)

if split_union_park_set != graph_node_set:
    raise ValueError(
        "Το union των split parks δεν ισούται με το graph node set. "
        f"graph_only={len(graph_node_set - split_union_park_set)}, "
        f"split_only={len(split_union_park_set - graph_node_set)}"
    )

print("Canonical NB11 input contract passed")
print("-" * 80)
print(f"Train shape            : {train_df.shape}")
print(f"Val shape              : {val_df.shape}")
print(f"Test shape             : {test_df.shape}")
print(f"Graph nodes            : {n_nodes}")
print(f"Directed edges         : {graph_edge_index.shape[1]}")
print(f"Self-loops             : {self_loops}")
print(f"Duplicate dir. edges   : {duplicate_directed_edges}")

display(graph_node_order_df.head())

Canonical NB11 input contract passed
--------------------------------------------------------------------------------
Train shape            : (1982736, 48)
Val shape              : (182998, 48)
Test shape             : (1086336, 48)
Graph nodes            : 256
Directed edges         : 1068
Self-loops             : 0
Duplicate dir. edges   : 0


,park_id,node_idx
0,00011,0
1,00090,1
2,00161,2
3,00164,3
4,00183,4


## Policy για feature roles και node-feature roles

Το NB11 κρατά ρητό separation μεταξύ:

### 1. Backbone / control columns
Στήλες που ορίζουν το canonical key-space ή downstream control context και **δεν** εισέρχονται ως learned node features:

- `park_id`
- `timestamp`
- `test_flag`
- `node_idx`

### 2. Target column
Η μεταβλητή πρόβλεψης:

- `Power_Output_Normalized`

### 3. Control reference column
Χρήσιμη ως auxiliary reference, αλλά όχι ως graph-model input by default:

- `Baseline_Prediction`

### 4. Static node features
Node-level χαρακτηριστικά που θεωρούνται σταθερά μέσα στον χρόνο και επαναχρησιμοποιούνται σε όλα τα snapshots.

Προτεραιότητα δίνεται σε numeric static fields όπως:

- `lat`
- `long`
- `hub_height_m`
- `rotor_diameter_m`
- `nominal_power_kW`

### 5. Dynamic node features
Όλα τα numeric candidate features που δεν είναι backbone / target / control reference / static features.

Αυτό περιλαμβάνει κατά κανόνα:

- exogenous meteorological inputs,
- temporal encodings,
- lag features,
- causal rolling features.

### 6. Blocked / excluded columns
Στήλες που υπάρχουν στο schema αλλά δεν περνούν στο tensor feature space στο NB11, π.χ.:

- identifiers,
- target,
- baseline reference,
- μη αριθμητικά static metadata όπως `turbine`,
- άλλα non-numeric candidate columns αν υπάρχουν.

Η παραπάνω επιλογή είναι **συντηρητική και benchmark-safe**.

In [3]:
# ============================================================
# Feature-role policy
# ============================================================

schema_columns = list(train_df.columns)

BACKBONE_CONTROL_COLUMNS = [
    col for col in [PARK_ID_COLUMN, TIMESTAMP_COLUMN, TEST_FLAG_COLUMN, NODE_IDX_COLUMN]
    if col in schema_columns
]

TARGET_COLUMNS = [TARGET_COLUMN]
CONTROL_REFERENCE_COLUMNS = [col for col in [BASELINE_COLUMN] if col in schema_columns]

STATIC_NUMERIC_FEATURE_COLUMNS = [
    col
    for col in STATIC_NUMERIC_CANDIDATES
    if col in schema_columns and pd.api.types.is_numeric_dtype(train_df[col])
]

STATIC_NON_NUMERIC_BLOCKED_COLUMNS = [
    col for col in STATIC_NON_NUMERIC_CANDIDATES
    if col in schema_columns
]

ALL_EXCLUDED_BASE = set(
    BACKBONE_CONTROL_COLUMNS
    + TARGET_COLUMNS
    + CONTROL_REFERENCE_COLUMNS
    + STATIC_NON_NUMERIC_BLOCKED_COLUMNS
)

NON_NUMERIC_OTHER_COLUMNS = [
    col
    for col in schema_columns
    if col not in ALL_EXCLUDED_BASE and not pd.api.types.is_numeric_dtype(train_df[col])
]

DYNAMIC_NODE_FEATURE_COLUMNS = [
    col
    for col in schema_columns
    if (
        col not in ALL_EXCLUDED_BASE
        and col not in set(STATIC_NUMERIC_FEATURE_COLUMNS)
        and col not in set(NON_NUMERIC_OTHER_COLUMNS)
        and pd.api.types.is_numeric_dtype(train_df[col])
    )
]

BLOCKED_EXCLUDED_COLUMNS = sorted(
    set(
        BACKBONE_CONTROL_COLUMNS
        + TARGET_COLUMNS
        + CONTROL_REFERENCE_COLUMNS
        + STATIC_NON_NUMERIC_BLOCKED_COLUMNS
        + NON_NUMERIC_OTHER_COLUMNS
    )
)

if len(DYNAMIC_NODE_FEATURE_COLUMNS) == 0:
    raise ValueError("Δεν βρέθηκαν dynamic numeric node features για packaging.")

# Δεν κάνουμε hard fail αν λείπει κάποιο static candidate.
# Αν όμως δεν υπάρχει κανένα static numeric feature, το σημειώνουμε ρητά.
HAS_STATIC_NUMERIC_FEATURES = len(STATIC_NUMERIC_FEATURE_COLUMNS) > 0


def classify_feature_role(col: str) -> str:
    """
    Χαρτογραφεί κάθε στήλη σε ένα σαφές feature role.
    """
    if col in BACKBONE_CONTROL_COLUMNS:
        return "backbone_control"
    if col == TARGET_COLUMN:
        return "target"
    if col == BASELINE_COLUMN:
        return "control_reference"
    if col in STATIC_NUMERIC_FEATURE_COLUMNS:
        return "static_node_feature"
    if col in DYNAMIC_NODE_FEATURE_COLUMNS:
        return "dynamic_node_feature"
    if col in STATIC_NON_NUMERIC_BLOCKED_COLUMNS:
        return "blocked_static_non_numeric"
    if col in NON_NUMERIC_OTHER_COLUMNS:
        return "blocked_non_numeric_other"
    return "excluded_other"


feature_role_manifest_df = pd.DataFrame(
    {
        "column": schema_columns,
        "role": [classify_feature_role(col) for col in schema_columns],
        "dtype_train": [str(train_df[col].dtype) for col in schema_columns],
        "dtype_val": [str(val_df[col].dtype) for col in schema_columns],
        "dtype_test": [str(test_df[col].dtype) for col in schema_columns],
        "numeric_in_train": [pd.api.types.is_numeric_dtype(train_df[col]) for col in schema_columns],
    }
)

node_feature_manifest_rows = []

for col in STATIC_NUMERIC_FEATURE_COLUMNS:
    node_feature_manifest_rows.append(
        {
            "column": col,
            "group": "static",
            "included_in_tensor": True,
            "dtype": str(train_df[col].dtype),
            "notes": "numeric static node feature",
        }
    )

for col in DYNAMIC_NODE_FEATURE_COLUMNS:
    node_feature_manifest_rows.append(
        {
            "column": col,
            "group": "dynamic",
            "included_in_tensor": True,
            "dtype": str(train_df[col].dtype),
            "notes": "numeric dynamic node feature",
        }
    )

for col in BLOCKED_EXCLUDED_COLUMNS:
    node_feature_manifest_rows.append(
        {
            "column": col,
            "group": "excluded",
            "included_in_tensor": False,
            "dtype": str(train_df[col].dtype),
            "notes": "excluded by NB11 packaging policy",
        }
    )

node_feature_manifest_df = (
    pd.DataFrame(node_feature_manifest_rows)
    .drop_duplicates(subset=["column"])
    .sort_values(["included_in_tensor", "group", "column"], ascending=[False, True, True])
    .reset_index(drop=True)
)

print("BACKBONE_CONTROL_COLUMNS:")
print(BACKBONE_CONTROL_COLUMNS)
print()

print("STATIC_NUMERIC_FEATURE_COLUMNS:")
print(STATIC_NUMERIC_FEATURE_COLUMNS)
print()

print("DYNAMIC_NODE_FEATURE_COLUMNS:")
print(DYNAMIC_NODE_FEATURE_COLUMNS)
print()

print("BLOCKED_EXCLUDED_COLUMNS:")
print(BLOCKED_EXCLUDED_COLUMNS)
print()

print("HAS_STATIC_NUMERIC_FEATURES:", HAS_STATIC_NUMERIC_FEATURES)

display(feature_role_manifest_df)
display(node_feature_manifest_df)

BACKBONE_CONTROL_COLUMNS:
['park_id', 'timestamp', 'test_flag', 'node_idx']

STATIC_NUMERIC_FEATURE_COLUMNS:
['lat', 'long', 'hub_height_m', 'rotor_diameter_m', 'nominal_power_kW']

DYNAMIC_NODE_FEATURE_COLUMNS:
['nwp_fcst_horiz_hours', 'T_HAG_2_M', 'RELHUM_HAG_2_M', 'PS_SFC_0_M', 'U_GVL_58_HL', 'V_GVL_58_HL', 'U_GVL_60_HL', 'V_GVL_60_HL', 'ASWDIFDS_SFC_0_M', 'ASWDIRS_SFC_0_M', 'U_GVL_58_HL_m1', 'V_GVL_58_HL_m1', 'U_GVL_60_HL_m1', 'V_GVL_60_HL_m1', 'U_GVL_58_HL_p1', 'V_GVL_58_HL_p1', 'U_GVL_60_HL_p1', 'V_GVL_60_HL_p1', 'ws_ref', 'Wind_Speed_100m_ms', 'hour', 'month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'Power_Output_Normalized_lag_1', 'Power_Output_Normalized_lag_3', 'Power_Output_Normalized_lag_6', 'Wind_Speed_100m_ms_lag_1', 'Wind_Speed_100m_ms_lag_3', 'Wind_Speed_100m_ms_lag_6', 'power_rolling_mean_6h', 'power_rolling_std_6h', 'wind_rolling_mean_6h', 'wind_rolling_std_6h']

BLOCKED_EXCLUDED_COLUMNS:
['Baseline_Prediction', 'Power_Output_Normalized', 'node_idx', 'park_i

,column,role,dtype_train,dtype_val,dtype_test,numeric_in_train
0,park_id,backbone_control,string,string,string,False
1,timestamp,backbone_control,datetime64[us],datetime64[us],datetime64[us],False
2,test_flag,backbone_control,int64,int64,int64,True
3,Power_Output_Normalized,target,float64,float64,float64,True
4,Baseline_Prediction,control_reference,float64,float64,float64,True
5,turbine,blocked_static_non_numeric,str,str,str,False
6,hub_height_m,static_node_feature,int64,int64,int64,True
7,rotor_diameter_m,static_node_feature,float64,float64,float64,True
8,nominal_power_kW,static_node_feature,float64,float64,float64,True
9,lat,static_node_feature,float64,float64,float64,True


,column,group,included_in_tensor,dtype,notes
0,ASWDIFDS_SFC_0_M,dynamic,True,float64,numeric dynamic node feature
1,ASWDIRS_SFC_0_M,dynamic,True,float64,numeric dynamic node feature
2,PS_SFC_0_M,dynamic,True,float64,numeric dynamic node feature
3,Power_Output_Normalized_lag_1,dynamic,True,float64,numeric dynamic node feature
4,Power_Output_Normalized_lag_3,dynamic,True,float64,numeric dynamic node feature
5,Power_Output_Normalized_lag_6,dynamic,True,float64,numeric dynamic node feature
6,RELHUM_HAG_2_M,dynamic,True,float64,numeric dynamic node feature
7,T_HAG_2_M,dynamic,True,float64,numeric dynamic node feature
8,U_GVL_58_HL,dynamic,True,float64,numeric dynamic node feature
9,U_GVL_58_HL_m1,dynamic,True,float64,numeric dynamic node feature


## Packaging schema

Κάθε split θα μετασχηματιστεί σε ένα serialized package με την ακόλουθη λογική:

### Graph-level shared artifacts
- `edge_index` : shape `[2, E]`
- `edge_attr_km` : shape `[E, 1]`
- `node_ids` : ordered according to `graph_node_order.csv`

### Node-level static tensor
- `static_x` : shape `[N, S]`

όπου:
- `N` = αριθμός nodes
- `S` = πλήθος static node features

### Time-indexed dynamic tensors
- `dynamic_x` : shape `[T, N, D]`
- `target_y` : shape `[T, N]`
- `baseline_reference` : shape `[T, N]`

όπου:
- `T` = αριθμός timestamps του split
- `D` = πλήθος dynamic node features

### Metadata
- `timestamps`
- `dynamic_feature_names`
- `static_feature_names`
- `target_name`
- `baseline_name`
- `split_name`

## Γιατί αυτή η μορφή είναι κατάλληλη

Η μορφή αυτή είναι ταυτόχρονα:

- deterministic,
- compact,
- model-ready,
- και εύκολα μετατρέψιμη σε per-snapshot `PyG Data` objects.

Το NB11 αποφεύγει να αποθηκεύσει εξαρχής πλήρες list από όλα τα `Data` objects, επειδή αυτό συνήθως είναι πιο βαρύ και λιγότερο αποδοτικό ως canonical packaging format.

In [4]:
# ============================================================
# Static consistency audit πάνω στο πλήρες split union
# ============================================================

combined_df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)

static_audit_cols = STATIC_NUMERIC_FEATURE_COLUMNS + STATIC_NON_NUMERIC_BLOCKED_COLUMNS

if len(static_audit_cols) == 0:
    print("Σημείωση: δεν βρέθηκαν static columns για explicit static audit.")
else:
    per_park_static_nunique = (
        combined_df.groupby(PARK_ID_COLUMN, sort=False)[static_audit_cols]
        .nunique(dropna=False)
    )

    bad_static_cells = []
    for park_id, row in per_park_static_nunique.iterrows():
        for col, nunique_value in row.items():
            if int(nunique_value) > 1:
                bad_static_cells.append((park_id, col, int(nunique_value)))

    if bad_static_cells:
        raise ValueError(
            "Βρέθηκαν non-static fields μέσα στα static candidates. "
            f"Examples={bad_static_cells[:10]}"
        )


# ============================================================
# Canonical static node frame με ordering από graph_node_order
# ============================================================

if HAS_STATIC_NUMERIC_FEATURES:
    node_static_unique = (
        combined_df[[PARK_ID_COLUMN] + STATIC_NUMERIC_FEATURE_COLUMNS]
        .drop_duplicates(subset=[PARK_ID_COLUMN])
        .copy()
    )

    node_static_frame = (
        graph_node_order_df[[PARK_ID_COLUMN, NODE_IDX_COLUMN]]
        .merge(node_static_unique, on=PARK_ID_COLUMN, how="left", validate="one_to_one")
        .sort_values(NODE_IDX_COLUMN, kind="mergesort")
        .reset_index(drop=True)
    )

    if node_static_frame[STATIC_NUMERIC_FEATURE_COLUMNS].isnull().any().any():
        raise ValueError(
            "Βρέθηκαν null values στα static numeric features μετά το canonical node merge."
        )

    static_x_np = node_static_frame[STATIC_NUMERIC_FEATURE_COLUMNS].to_numpy(dtype=np.float32)
else:
    node_static_frame = graph_node_order_df[[PARK_ID_COLUMN, NODE_IDX_COLUMN]].copy()
    static_x_np = np.empty((n_nodes, 0), dtype=np.float32)

static_x_tensor = torch.as_tensor(static_x_np, dtype=torch.float32)


# ============================================================
# Edge attributes από canonical distance matrix
# ============================================================

edge_index_tensor = torch.as_tensor(graph_edge_index, dtype=torch.long)

src_idx = graph_edge_index[0]
dst_idx = graph_edge_index[1]

edge_attr_km_np = graph_distance_matrix_km[src_idx, dst_idx].reshape(-1, 1).astype(np.float32)
edge_attr_km_tensor = torch.as_tensor(edge_attr_km_np, dtype=torch.float32)

if edge_attr_km_tensor.shape[0] != edge_index_tensor.shape[1]:
    raise ValueError("Το edge_attr_km δεν συμφωνεί σε πλήθος edges με το edge_index.")


# ============================================================
# Coverage-aware packaging helper
# ============================================================

def build_split_graph_package(split_df: pd.DataFrame, split_name: str) -> dict[str, Any]:
    """
    Μετασχηματίζει canonical split dataframe σε graph-ready packaged dataset.

    Κρίσιμη μεθοδολογική σημείωση:
    - Δεν υποθέτουμε πλέον πλήρες node coverage σε κάθε timestamp.
    - Χτίζουμε dense tensors shape [T, N, ...] με NaN fill.
    - Παράλληλα κρατάμε observed_mask ώστε το downstream modeling stage
      να γνωρίζει ποια (timestamp, node) entries είναι πραγματικά observed.
    """
    work_df = (
        split_df.sort_values([TIMESTAMP_COLUMN, NODE_IDX_COLUMN], kind="mergesort")
        .reset_index(drop=True)
        .copy()
    )

    ordered_timestamps = sorted(work_df[TIMESTAMP_COLUMN].drop_duplicates().tolist())
    n_timestamps = len(ordered_timestamps)
    n_dynamic = len(DYNAMIC_NODE_FEATURE_COLUMNS)

    if n_timestamps == 0:
        raise ValueError(f"Το split `{split_name}` είναι κενό μετά το canonical sorting.")

    timestamp_to_idx = {ts: i for i, ts in enumerate(ordered_timestamps)}

    # ------------------------------------------------------------
    # Dense tensors με NaN fill
    # ------------------------------------------------------------
    dynamic_x_np = np.full((n_timestamps, n_nodes, n_dynamic), np.nan, dtype=np.float32)
    target_y_np = np.full((n_timestamps, n_nodes), np.nan, dtype=np.float32)
    baseline_reference_np = np.full((n_timestamps, n_nodes), np.nan, dtype=np.float32)

    # Boolean mask που δείχνει αν υπάρχει πραγματική παρατήρηση
    observed_mask_np = np.zeros((n_timestamps, n_nodes), dtype=bool)

    # ------------------------------------------------------------
    # Vectorized indexing
    # ------------------------------------------------------------
    t_idx = work_df[TIMESTAMP_COLUMN].map(timestamp_to_idx).to_numpy(dtype=np.int64)
    n_idx = work_df[NODE_IDX_COLUMN].to_numpy(dtype=np.int64)

    dynamic_vals = work_df[DYNAMIC_NODE_FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    target_vals = work_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
    baseline_vals = work_df[BASELINE_COLUMN].to_numpy(dtype=np.float32)

    dynamic_x_np[t_idx, n_idx, :] = dynamic_vals
    target_y_np[t_idx, n_idx] = target_vals
    baseline_reference_np[t_idx, n_idx] = baseline_vals
    observed_mask_np[t_idx, n_idx] = True

    # ------------------------------------------------------------
    # Coverage audit ανά timestamp
    # ------------------------------------------------------------
    node_count_per_timestamp = observed_mask_np.sum(axis=1)
    full_coverage_mask = node_count_per_timestamp == n_nodes
    partial_timestamp_count = int((~full_coverage_mask).sum())
    full_coverage_timestamp_count = int(full_coverage_mask.sum())

    if np.any(node_count_per_timestamp == 0):
        raise ValueError(
            f"Το split `{split_name}` περιέχει timestamp χωρίς καμία observed node entry."
        )

    coverage_summary_df = pd.DataFrame(
        {
            "timestamp": [pd.Timestamp(ts).strftime("%Y-%m-%d %H:%M:%S") for ts in ordered_timestamps],
            "observed_node_count": node_count_per_timestamp.astype(int),
            "missing_node_count": (n_nodes - node_count_per_timestamp).astype(int),
            "full_node_coverage": full_coverage_mask,
        }
    )

    package = {
        "split_name": split_name,
        "node_ids": graph_node_order_df[PARK_ID_COLUMN].tolist(),
        "timestamps": [pd.Timestamp(ts).strftime("%Y-%m-%d %H:%M:%S") for ts in ordered_timestamps],
        "edge_index": edge_index_tensor.clone(),
        "edge_attr_km": edge_attr_km_tensor.clone(),
        "static_x": torch.as_tensor(static_x_np, dtype=torch.float32),
        "dynamic_x": torch.as_tensor(dynamic_x_np, dtype=torch.float32),
        "target_y": torch.as_tensor(target_y_np, dtype=torch.float32),
        "baseline_reference": torch.as_tensor(baseline_reference_np, dtype=torch.float32),
        "observed_mask": torch.as_tensor(observed_mask_np, dtype=torch.bool),
        "static_feature_names": STATIC_NUMERIC_FEATURE_COLUMNS.copy(),
        "dynamic_feature_names": DYNAMIC_NODE_FEATURE_COLUMNS.copy(),
        "target_name": TARGET_COLUMN,
        "baseline_name": BASELINE_COLUMN,
        "n_nodes": n_nodes,
        "n_timestamps": n_timestamps,
        "n_static_features": static_x_np.shape[1],
        "n_dynamic_features": n_dynamic,
        "partial_timestamp_count": partial_timestamp_count,
        "full_coverage_timestamp_count": full_coverage_timestamp_count,
        "coverage_summary_df": coverage_summary_df,
    }

    return package


# ============================================================
# PyG snapshot helper
# ============================================================

def build_pyg_snapshot(package: dict[str, Any], t_index: int) -> Any:
    """
    Μετατρέπει packaged split + χρονικό index σε PyG Data snapshot.

    Σημείωση:
    Το snapshot κρατά και observed_mask για να είναι explicit ποιες node
    entries είναι observed στο συγκεκριμένο timestamp.
    """
    if not TORCH_PYG_AVAILABLE:
        raise ImportError("Το torch_geometric δεν είναι διαθέσιμο στο current environment.")

    if not (0 <= t_index < package["n_timestamps"]):
        raise IndexError(
            f"t_index out of range: {t_index} not in [0, {package['n_timestamps'] - 1}]"
        )

    # Το node feature matrix είναι concat(static_x, dynamic_x[t]).
    x = torch.cat(
        [package["static_x"], package["dynamic_x"][t_index]],
        dim=1,
    )

    data = Data(
        x=x,
        edge_index=package["edge_index"],
        edge_attr=package["edge_attr_km"],
        y=package["target_y"][t_index],
    )

    data.num_nodes = package["n_nodes"]
    data.timestamp = package["timestamps"][t_index]
    data.node_ids = package["node_ids"]
    data.static_feature_names = package["static_feature_names"]
    data.dynamic_feature_names = package["dynamic_feature_names"]
    data.target_name = package["target_name"]
    data.observed_mask = package["observed_mask"][t_index].clone()

    return data


print("Static + edge packaging helpers ready")
print("-" * 80)
print(f"Static tensor shape : {tuple(static_x_tensor.shape)}")
print(f"Edge index shape    : {tuple(edge_index_tensor.shape)}")
print(f"Edge attr shape     : {tuple(edge_attr_km_tensor.shape)}")

Static + edge packaging helpers ready
--------------------------------------------------------------------------------
Static tensor shape : (256, 5)
Edge index shape    : (2, 1068)
Edge attr shape     : (1068, 1)


## Export policy του NB11

Το notebook γράφει intentional outputs μόνο κάτω από dedicated packaging directory.

### CSV manifests
- `nb11_feature_role_manifest.csv`
- `nb11_node_feature_manifest.csv`
- `nb11_split_graph_packaging_summary.csv`
- `nb11_packaging_status_manifest.csv`

### Serialized packages
- `train_graph_dataset.pt`
- `val_graph_dataset.pt`
- `test_graph_dataset.pt`

### Coverage-aware packaging note
Το NB11 **δεν** υποθέτει πλέον ότι κάθε timestamp έχει πλήρες node coverage.

Για αυτόν τον λόγο, κάθε serialized package περιλαμβάνει επίσης:

- `observed_mask`
- `coverage_summary_df`
- `partial_timestamp_count`
- `full_coverage_timestamp_count`

ώστε το επόμενο graph-based stage να γνωρίζει ρητά πού υπάρχουν πραγματικές παρατηρήσεις και πού όχι.

### Optional preview
Αν το `torch_geometric` είναι διαθέσιμο, εξάγεται και preview bundle με first-snapshot `Data` objects για deterministic inspection.

## Σημαντική σημείωση

Τα exported `.pt` packages είναι **local rerun artifacts** και όχι benchmark artifacts.

Το canonical benchmark artifact του repository παραμένει ανεπηρέαστο.

In [5]:
# ============================================================
# Build packaged datasets (hardened / freeze-safer version)
# ============================================================

# Χτίζουμε πρώτα τα in-memory packages από τα canonical splits.
# Σημαντικό:
# - κρατάμε coverage-aware λογική,
# - ΔΕΝ αλλάζουμε benchmark artifacts,
# - ΔΕΝ κάνουμε training,
# - και απλώς σκληραίνουμε το export layer.
train_package = build_split_graph_package(train_df, "train")
val_package = build_split_graph_package(val_df, "val")
test_package = build_split_graph_package(test_df, "test")


# ============================================================
# Helper: observed-mask integrity checks
# ============================================================

def assert_observed_entries_are_valid(package: dict[str, Any]) -> None:
    """
    Ελέγχει ότι όπου observed_mask == True:
    - το target δεν είναι NaN
    - το baseline reference δεν είναι NaN
    - τα dynamic features δεν είναι NaN
    """
    split_name = package["split_name"]

    observed_mask = package["observed_mask"].cpu()
    target_y = package["target_y"].cpu()
    baseline_reference = package["baseline_reference"].cpu()
    dynamic_x = package["dynamic_x"].cpu()

    # --------------------------------------------------------
    # 1) observed entries στο target
    # --------------------------------------------------------
    target_missing_on_observed = torch.isnan(target_y)[observed_mask]
    if target_missing_on_observed.any():
        raise ValueError(
            f"Το split `{split_name}` έχει observed entries με NaN target_y."
        )

    # --------------------------------------------------------
    # 2) observed entries στο baseline reference
    # --------------------------------------------------------
    baseline_missing_on_observed = torch.isnan(baseline_reference)[observed_mask]
    if baseline_missing_on_observed.any():
        raise ValueError(
            f"Το split `{split_name}` έχει observed entries με NaN baseline_reference."
        )

    # --------------------------------------------------------
    # 3) observed entries στα dynamic features
    # dynamic_x shape = [T, N, D]
    # observed_mask shape = [T, N]
    # Για να κάνουμε broadcast-safe έλεγχο, το επεκτείνουμε σε [T, N, 1].
    # --------------------------------------------------------
    if dynamic_x.numel() > 0:
        observed_mask_3d = observed_mask.unsqueeze(-1).expand_as(dynamic_x)
        dynamic_missing_on_observed = torch.isnan(dynamic_x)[observed_mask_3d]
        if dynamic_missing_on_observed.any():
            raise ValueError(
                f"Το split `{split_name}` έχει observed entries με NaN dynamic_x values."
            )


# ============================================================
# Helper: remove non-portable pandas objects πριν από το torch.save
# ============================================================

def make_serializable_package(package: dict[str, Any]) -> dict[str, Any]:
    """
    Επιστρέφει portable version του package.

    Κρατάμε:
    - tensors
    - Python primitives / lists / dicts

    Δεν κρατάμε:
    - pandas DataFrame objects μέσα στο .pt
    γιατί αυτό κάνει το artifact πιο brittle σε cross-environment loading.
    """
    serializable = {
        key: value
        for key, value in package.items()
        if key != "coverage_summary_df"
    }

    # Προαιρετικά κρατάμε compact metadata για να μη χαθεί η πληροφορία
    # του coverage μέσα στο serialized artifact.
    coverage_df = package["coverage_summary_df"].copy()
    serializable["coverage_summary_records"] = coverage_df.to_dict(orient="records")
    serializable["coverage_summary_columns"] = list(coverage_df.columns)

    return serializable


# ============================================================
# Integrity checks πριν από οποιοδήποτε export
# ============================================================

for pkg in [train_package, val_package, test_package]:
    assert_observed_entries_are_valid(pkg)

print("Observed-mask integrity checks passed.")


# ============================================================
# Local CSV exports για timestamp-level coverage inspection
# ============================================================

NB11_TRAIN_TIMESTAMP_COVERAGE_PATH = EXPORT_DIR / "nb11_train_timestamp_coverage.csv"
NB11_VAL_TIMESTAMP_COVERAGE_PATH = EXPORT_DIR / "nb11_val_timestamp_coverage.csv"
NB11_TEST_TIMESTAMP_COVERAGE_PATH = EXPORT_DIR / "nb11_test_timestamp_coverage.csv"

train_package["coverage_summary_df"].to_csv(NB11_TRAIN_TIMESTAMP_COVERAGE_PATH, index=False)
val_package["coverage_summary_df"].to_csv(NB11_VAL_TIMESTAMP_COVERAGE_PATH, index=False)
test_package["coverage_summary_df"].to_csv(NB11_TEST_TIMESTAMP_COVERAGE_PATH, index=False)


# ============================================================
# Split-wise packaging summary
# ============================================================

split_graph_packaging_summary_df = pd.DataFrame(
    [
        {
            "split": pkg["split_name"],
            "n_rows_original": int(len(df)),
            "n_timestamps": pkg["n_timestamps"],
            "n_nodes": pkg["n_nodes"],
            "n_static_features": pkg["n_static_features"],
            "n_dynamic_features": pkg["n_dynamic_features"],
            "full_coverage_timestamp_count": pkg["full_coverage_timestamp_count"],
            "partial_timestamp_count": pkg["partial_timestamp_count"],
            "observed_ratio": float(pkg["observed_mask"].sum().item()) / float(pkg["observed_mask"].numel()),
            "dynamic_x_shape": str(tuple(pkg["dynamic_x"].shape)),
            "static_x_shape": str(tuple(pkg["static_x"].shape)),
            "target_y_shape": str(tuple(pkg["target_y"].shape)),
            "baseline_reference_shape": str(tuple(pkg["baseline_reference"].shape)),
            "observed_mask_shape": str(tuple(pkg["observed_mask"].shape)),
            "edge_index_shape": str(tuple(pkg["edge_index"].shape)),
            "edge_attr_km_shape": str(tuple(pkg["edge_attr_km"].shape)),
            "timestamp_min": pkg["timestamps"][0],
            "timestamp_max": pkg["timestamps"][-1],
        }
        for pkg, df in [
            (train_package, train_df),
            (val_package, val_df),
            (test_package, test_df),
        ]
    ]
)

display(split_graph_packaging_summary_df)


# ============================================================
# Export manifests
# ============================================================

feature_role_manifest_df.to_csv(NB11_FEATURE_ROLE_MANIFEST_PATH, index=False)
node_feature_manifest_df.to_csv(NB11_NODE_FEATURE_MANIFEST_PATH, index=False)
split_graph_packaging_summary_df.to_csv(NB11_SPLIT_GRAPH_PACKAGING_SUMMARY_PATH, index=False)


# ============================================================
# Portable serialized packages
# ============================================================

train_package_serializable = make_serializable_package(train_package)
val_package_serializable = make_serializable_package(val_package)
test_package_serializable = make_serializable_package(test_package)

if WRITE_SERIALIZED_PACKAGES:
    torch.save(train_package_serializable, NB11_TRAIN_GRAPH_DATASET_PATH)
    torch.save(val_package_serializable, NB11_VAL_GRAPH_DATASET_PATH)
    torch.save(test_package_serializable, NB11_TEST_GRAPH_DATASET_PATH)


# ============================================================
# Immediate reload check
# Στόχος: να μη λέμε μόνο "γράφτηκαν", αλλά και "φορτώνονται".
# ============================================================

def assert_reloadable_package(package_path: Path, expected_split_name: str) -> None:
    reloaded = torch.load(package_path, map_location="cpu")

    required_keys = [
        "split_name",
        "node_ids",
        "timestamps",
        "edge_index",
        "edge_attr_km",
        "static_x",
        "dynamic_x",
        "target_y",
        "baseline_reference",
        "observed_mask",
        "static_feature_names",
        "dynamic_feature_names",
        "target_name",
        "baseline_name",
        "n_nodes",
        "n_timestamps",
        "n_static_features",
        "n_dynamic_features",
        "partial_timestamp_count",
        "full_coverage_timestamp_count",
    ]

    missing_keys = [key for key in required_keys if key not in reloaded]
    if missing_keys:
        raise ValueError(
            f"Το reloaded package `{expected_split_name}` λείπει required keys: {missing_keys}"
        )

    if reloaded["split_name"] != expected_split_name:
        raise ValueError(
            f"Λάθος split_name στο reloaded package: "
            f"expected `{expected_split_name}`, got `{reloaded['split_name']}`"
        )

    if tuple(reloaded["observed_mask"].shape) != tuple(reloaded["target_y"].shape):
        raise ValueError(
            f"Shape mismatch στο `{expected_split_name}`: "
            f"observed_mask {tuple(reloaded['observed_mask'].shape)} vs "
            f"target_y {tuple(reloaded['target_y'].shape)}"
        )

    if reloaded["dynamic_x"].shape[:2] != reloaded["target_y"].shape:
        raise ValueError(
            f"Shape mismatch στο `{expected_split_name}`: "
            f"dynamic_x[:2] {tuple(reloaded['dynamic_x'].shape[:2])} vs "
            f"target_y {tuple(reloaded['target_y'].shape)}"
        )

    if reloaded["baseline_reference"].shape != reloaded["target_y"].shape:
        raise ValueError(
            f"Shape mismatch στο `{expected_split_name}`: "
            f"baseline_reference {tuple(reloaded['baseline_reference'].shape)} vs "
            f"target_y {tuple(reloaded['target_y'].shape)}"
        )


if WRITE_SERIALIZED_PACKAGES:
    assert_reloadable_package(NB11_TRAIN_GRAPH_DATASET_PATH, "train")
    assert_reloadable_package(NB11_VAL_GRAPH_DATASET_PATH, "val")
    assert_reloadable_package(NB11_TEST_GRAPH_DATASET_PATH, "test")
    print("Portable reload checks passed.")


# ============================================================
# Optional PyG preview export
# Το preview μπορεί να βασιστεί στο original in-memory package.
# Δεν το χρησιμοποιούμε ως canonical training artifact.
# ============================================================

preview_exported = False

if EXPORT_PYG_PREVIEW_OBJECTS and TORCH_PYG_AVAILABLE:
    preview_bundle = {
        "train_first_snapshot": build_pyg_snapshot(train_package, 0),
        "val_first_snapshot": build_pyg_snapshot(val_package, 0),
        "test_first_snapshot": build_pyg_snapshot(test_package, 0),
    }
    torch.save(preview_bundle, NB11_PREVIEW_PYG_OBJECTS_PATH)
    preview_exported = True


# ============================================================
# Packaging status manifest
# Καλύτερα paths σε POSIX style για λίγο πιο καθαρό portability.
# ============================================================

packaging_status_manifest_df = pd.DataFrame(
    [
        {
            "artifact": "nb11_feature_role_manifest.csv",
            "written": NB11_FEATURE_ROLE_MANIFEST_PATH.exists(),
            "path": NB11_FEATURE_ROLE_MANIFEST_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_node_feature_manifest.csv",
            "written": NB11_NODE_FEATURE_MANIFEST_PATH.exists(),
            "path": NB11_NODE_FEATURE_MANIFEST_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_split_graph_packaging_summary.csv",
            "written": NB11_SPLIT_GRAPH_PACKAGING_SUMMARY_PATH.exists(),
            "path": NB11_SPLIT_GRAPH_PACKAGING_SUMMARY_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "train_graph_dataset.pt",
            "written": NB11_TRAIN_GRAPH_DATASET_PATH.exists(),
            "path": NB11_TRAIN_GRAPH_DATASET_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "val_graph_dataset.pt",
            "written": NB11_VAL_GRAPH_DATASET_PATH.exists(),
            "path": NB11_VAL_GRAPH_DATASET_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "test_graph_dataset.pt",
            "written": NB11_TEST_GRAPH_DATASET_PATH.exists(),
            "path": NB11_TEST_GRAPH_DATASET_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_preview_pyg_objects.pt",
            "written": bool(preview_exported and NB11_PREVIEW_PYG_OBJECTS_PATH.exists()),
            "path": NB11_PREVIEW_PYG_OBJECTS_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_train_timestamp_coverage.csv",
            "written": NB11_TRAIN_TIMESTAMP_COVERAGE_PATH.exists(),
            "path": NB11_TRAIN_TIMESTAMP_COVERAGE_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_val_timestamp_coverage.csv",
            "written": NB11_VAL_TIMESTAMP_COVERAGE_PATH.exists(),
            "path": NB11_VAL_TIMESTAMP_COVERAGE_PATH.relative_to(ROOT).as_posix(),
        },
        {
            "artifact": "nb11_test_timestamp_coverage.csv",
            "written": NB11_TEST_TIMESTAMP_COVERAGE_PATH.exists(),
            "path": NB11_TEST_TIMESTAMP_COVERAGE_PATH.relative_to(ROOT).as_posix(),
        },
    ]
)

packaging_status_manifest_df.to_csv(NB11_PACKAGING_STATUS_MANIFEST_PATH, index=False)

print("NB11 exports written successfully")
print("-" * 80)
display(packaging_status_manifest_df)

Observed-mask integrity checks passed.


,split,n_rows_original,n_timestamps,n_nodes,n_static_features,n_dynamic_features,full_coverage_timestamp_count,partial_timestamp_count,observed_ratio,dynamic_x_shape,static_x_shape,target_y_shape,baseline_reference_shape,observed_mask_shape,edge_index_shape,edge_attr_km_shape,timestamp_min,timestamp_max
0,train,1982736,7819,256,5,36,1325,6494,0.990544,"(7819, 256, 36)","(256, 5)","(7819, 256)","(7819, 256)","(7819, 256)","(2, 1068)","(1068, 1)",2018-12-08 06:00:00,2019-11-01 00:00:00
1,val,182998,720,256,5,36,127,593,0.992828,"(720, 256, 36)","(256, 5)","(720, 256)","(720, 256)","(720, 256)","(2, 1068)","(1068, 1)",2019-11-01 01:00:00,2019-12-01 00:00:00
2,test,1086336,4295,256,5,36,199,4096,0.988009,"(4295, 256, 36)","(256, 5)","(4295, 256)","(4295, 256)","(4295, 256)","(2, 1068)","(1068, 1)",2019-12-01 01:00:00,2020-06-01 23:00:00


Portable reload checks passed.
NB11 exports written successfully
--------------------------------------------------------------------------------


,artifact,written,path
0,nb11_feature_role_manifest.csv,True,data/processed/graph_packaging/nb11_feature_ro...
1,nb11_node_feature_manifest.csv,True,data/processed/graph_packaging/nb11_node_featu...
2,nb11_split_graph_packaging_summary.csv,True,data/processed/graph_packaging/nb11_split_grap...
3,train_graph_dataset.pt,True,data/processed/graph_packaging/train_graph_dat...
4,val_graph_dataset.pt,True,data/processed/graph_packaging/val_graph_datas...
5,test_graph_dataset.pt,True,data/processed/graph_packaging/test_graph_data...
6,nb11_preview_pyg_objects.pt,True,data/processed/graph_packaging/nb11_preview_py...
7,nb11_train_timestamp_coverage.csv,True,data/processed/graph_packaging/nb11_train_time...
8,nb11_val_timestamp_coverage.csv,True,data/processed/graph_packaging/nb11_val_timest...
9,nb11_test_timestamp_coverage.csv,True,data/processed/graph_packaging/nb11_test_times...


## Συμπέρασμα

Το NB11 ολοκληρώνει το **graph-model input packaging** stage του canonical workflow.

Το notebook παρήγαγε:

- explicit feature-role manifests,
- explicit node-feature manifests,
- split-wise graph packaging summary,
- serialized graph-ready dataset packages για `train / val / test`,
- timestamp-level coverage exports,
- και optional PyG preview objects.

### Τι τεκμηριώνεται από το NB11

Το notebook τεκμηριώνει ότι τα canonical split και graph artifacts μπορούν να μετασχηματιστούν σε:

- reproducible graph-ready tensors,
- deterministic node-ordered split packages,
- PyG-ready snapshot objects,
- και **coverage-aware** graph inputs μέσω `observed_mask`.

### Γιατί αυτό είναι σημαντικό

Το current packaging stage έδειξε ότι τα canonical split artifacts δεν πρέπει να θεωρούνται αυτονόητα ως πλήρως rectangular node-by-time tensors.

Άρα το graph handoff του repository πρέπει να είναι:

- benchmark-safe,
- split-safe,
- contract-safe,
- και coverage-aware.

### Τι δεν τεκμηριώνεται από το NB11

Το notebook δεν τεκμηριώνει:

- graph-model training quality,
- GNN superiority,
- graph benchmark results,
- anomaly detection,
- PHM functionality,
- ή digital twin deployment.

Άρα το NB11 παραμένει ένα **strict packaging / handoff stage** για future graph-based forecasting work.

In [6]:
# ============================================================
# Final sanity checks
# ============================================================

required_written_files = [
    NB11_FEATURE_ROLE_MANIFEST_PATH,
    NB11_NODE_FEATURE_MANIFEST_PATH,
    NB11_SPLIT_GRAPH_PACKAGING_SUMMARY_PATH,
    NB11_PACKAGING_STATUS_MANIFEST_PATH,
    NB11_TRAIN_TIMESTAMP_COVERAGE_PATH,
    NB11_VAL_TIMESTAMP_COVERAGE_PATH,
    NB11_TEST_TIMESTAMP_COVERAGE_PATH,
]

if WRITE_SERIALIZED_PACKAGES:
    required_written_files.extend(
        [
            NB11_TRAIN_GRAPH_DATASET_PATH,
            NB11_VAL_GRAPH_DATASET_PATH,
            NB11_TEST_GRAPH_DATASET_PATH,
        ]
    )

missing_written_files = [path for path in required_written_files if not path.exists()]
if missing_written_files:
    raise FileNotFoundError(
        f"Λείπουν expected NB11 exports: {missing_written_files}"
    )

final_checks_df = pd.DataFrame(
    [
        {
            "check": "split_schema_identical",
            "status": list(train_df.columns) == list(val_df.columns) == list(test_df.columns),
            "details": "train / val / test share identical schema",
        },
        {
            "check": "split_union_equals_graph_nodes",
            "status": split_union_park_set == graph_node_set,
            "details": "all split parks are represented in graph_node_order",
        },
        {
            "check": "train_flag_contract",
            "status": train_flags == {0},
            "details": f"found={train_flags}",
        },
        {
            "check": "val_flag_contract",
            "status": val_flags == {0},
            "details": f"found={val_flags}",
        },
        {
            "check": "test_flag_contract",
            "status": test_flags == {1},
            "details": f"found={test_flags}",
        },
        {
            "check": "dynamic_features_non_empty",
            "status": len(DYNAMIC_NODE_FEATURE_COLUMNS) > 0,
            "details": f"n_dynamic_features={len(DYNAMIC_NODE_FEATURE_COLUMNS)}",
        },
        {
            "check": "train_package_shape_valid",
            "status": (
                train_package["dynamic_x"].ndim == 3
                and train_package["target_y"].ndim == 2
                and train_package["observed_mask"].ndim == 2
            ),
            "details": (
                f"dynamic_x={tuple(train_package['dynamic_x'].shape)}, "
                f"target_y={tuple(train_package['target_y'].shape)}, "
                f"observed_mask={tuple(train_package['observed_mask'].shape)}"
            ),
        },
        {
            "check": "val_package_shape_valid",
            "status": (
                val_package["dynamic_x"].ndim == 3
                and val_package["target_y"].ndim == 2
                and val_package["observed_mask"].ndim == 2
            ),
            "details": (
                f"dynamic_x={tuple(val_package['dynamic_x'].shape)}, "
                f"target_y={tuple(val_package['target_y'].shape)}, "
                f"observed_mask={tuple(val_package['observed_mask'].shape)}"
            ),
        },
        {
            "check": "test_package_shape_valid",
            "status": (
                test_package["dynamic_x"].ndim == 3
                and test_package["target_y"].ndim == 2
                and test_package["observed_mask"].ndim == 2
            ),
            "details": (
                f"dynamic_x={tuple(test_package['dynamic_x'].shape)}, "
                f"target_y={tuple(test_package['target_y'].shape)}, "
                f"observed_mask={tuple(test_package['observed_mask'].shape)}"
            ),
        },
        {
            "check": "observed_mask_matches_target_shape_train",
            "status": tuple(train_package["observed_mask"].shape) == tuple(train_package["target_y"].shape),
            "details": "train observed_mask shape matches target_y shape",
        },
        {
            "check": "observed_mask_matches_target_shape_val",
            "status": tuple(val_package["observed_mask"].shape) == tuple(val_package["target_y"].shape),
            "details": "val observed_mask shape matches target_y shape",
        },
        {
            "check": "observed_mask_matches_target_shape_test",
            "status": tuple(test_package["observed_mask"].shape) == tuple(test_package["target_y"].shape),
            "details": "test observed_mask shape matches target_y shape",
        },
        {
            "check": "train_mask_exists",
            "status": "observed_mask" in train_package,
            "details": f"partial_timestamps={train_package['partial_timestamp_count']}",
        },
        {
            "check": "val_mask_exists",
            "status": "observed_mask" in val_package,
            "details": f"partial_timestamps={val_package['partial_timestamp_count']}",
        },
        {
            "check": "test_mask_exists",
            "status": "observed_mask" in test_package,
            "details": f"partial_timestamps={test_package['partial_timestamp_count']}",
        },
        {
            "check": "edge_attr_matches_edge_count",
            "status": edge_attr_km_tensor.shape[0] == edge_index_tensor.shape[1],
            "details": f"edge_attr_rows={edge_attr_km_tensor.shape[0]}, edge_count={edge_index_tensor.shape[1]}",
        },
        {
            "check": "serialized_packages_written",
            "status": (
                True if not WRITE_SERIALIZED_PACKAGES
                else (
                    NB11_TRAIN_GRAPH_DATASET_PATH.exists()
                    and NB11_VAL_GRAPH_DATASET_PATH.exists()
                    and NB11_TEST_GRAPH_DATASET_PATH.exists()
                )
            ),
            "details": "graph-ready serialized packages available",
        },
        {
            "check": "baseline_metrics_modified_here",
            "status": False,
            "details": "expected False by NB11 scope",
        },
        {
            "check": "model_training_performed_here",
            "status": False,
            "details": "expected False by NB11 scope",
        },
    ]
)

display(final_checks_df)

hard_fail_checks = final_checks_df.loc[
    final_checks_df["check"].isin(
        [
            "split_schema_identical",
            "split_union_equals_graph_nodes",
            "train_flag_contract",
            "val_flag_contract",
            "test_flag_contract",
            "dynamic_features_non_empty",
            "train_package_shape_valid",
            "val_package_shape_valid",
            "test_package_shape_valid",
            "observed_mask_matches_target_shape_train",
            "observed_mask_matches_target_shape_val",
            "observed_mask_matches_target_shape_test",
            "train_mask_exists",
            "val_mask_exists",
            "test_mask_exists",
            "edge_attr_matches_edge_count",
            "serialized_packages_written",
        ]
    )
]

if not bool(hard_fail_checks["status"].all()):
    raise AssertionError(
        "Κάποιο hard NB11 packaging check απέτυχε. "
        "Δες το final_checks_df και τα exported manifests."
    )

print("\nNB11 PACKAGING CHECKS PASSED")
print("Το notebook ολοκληρώθηκε ως strict graph-model input packaging stage.")
print("Το packaging είναι coverage-aware και δεν υπέθεσε πλήρες node coverage σε κάθε timestamp.")
print("Δεν εκτελέστηκε training και δεν τροποποιήθηκε benchmark reporting.")

,check,status,details
0,split_schema_identical,True,train / val / test share identical schema
1,split_union_equals_graph_nodes,True,all split parks are represented in graph_node_...
2,train_flag_contract,True,found={0}
3,val_flag_contract,True,found={0}
4,test_flag_contract,True,found={1}
5,dynamic_features_non_empty,True,n_dynamic_features=36
6,train_package_shape_valid,True,"dynamic_x=(7819, 256, 36), target_y=(7819, 256..."
7,val_package_shape_valid,True,"dynamic_x=(720, 256, 36), target_y=(720, 256),..."
8,test_package_shape_valid,True,"dynamic_x=(4295, 256, 36), target_y=(4295, 256..."
9,observed_mask_matches_target_shape_train,True,train observed_mask shape matches target_y shape



NB11 PACKAGING CHECKS PASSED
Το notebook ολοκληρώθηκε ως strict graph-model input packaging stage.
Το packaging είναι coverage-aware και δεν υπέθεσε πλήρες node coverage σε κάθε timestamp.
Δεν εκτελέστηκε training και δεν τροποποιήθηκε benchmark reporting.
